# MODELING EXPERIMENTS

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

from fake_news_co.paths import DATA_PROCESSED # Export the path to the processed data directory

In [ ]:
df = pd.read_csv(DATA_PROCESSED / "dataset.csv")

le = LabelEncoder()

df["label_id"] = le.fit_transform(df["label"])   # 0,1,2 alfabético
# le.classes_ guarda el mapeo para la matriz de confusión y la demo
train = df[df["split"] == "train"][["text", "label_id"]]
val = df[df["split"] == "val"][["text", "label_id"]]
test = df[df["split"] == "test"][["text", "label_id"]]

print(f"Train shape: {train.shape}")
print(f"Validation shape: {val.shape}")
print(f"Test shape: {test.shape}")

X_train, y_train = train["text"], train["label_id"]
X_val, y_val = val["text"], val["label_id"]
X_test, y_test = test["text"], test["label_id"]

# 1. TF-IDF BASELINE

## 1.1 TF-IDF 

In [ ]:
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline


pipe_tfid = Pipeline(
    [("tfidf", TfidfVectorizer(min_df=2, ngram_range=(1, 2))), 
     ("clf", LogisticRegression(max_iter=100_000, random_state=42, class_weight="balanced", solver="lbfgs"))]
)

params = {
    "clf__C": [0.01, 0.1, 1, 10]
    }

grid_tfidf = GridSearchCV(
    pipe_tfid, 
    params, cv=3, scoring="f1_macro", n_jobs=-1, verbose=1
)
time_start = time.time()
grid_tfidf.fit(X_train, y_train)
time_end = time.time()
print(f"Grid search time: {time_end - time_start} seconds")

best_params = grid_tfidf.best_params_
print(f"Best parameters: {best_params}")

best_estimator = grid_tfidf.best_estimator_

In [ ]:
# F1-Macro on validation set

y_val_pred = best_estimator.predict(X_val)
val_f1 = f1_score(y_val, y_val_pred, average="macro")   # (verdad, predicción)
print(f"Macro-F1 on validation: {val_f1:.4f}")

from sklearn.metrics import classification_report
print(classification_report(y_val, y_val_pred, digits=3, target_names=le.classes_))

## 1.2 TF-IDF SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

pipe_smote = ImbPipeline([
    ("tfidf", TfidfVectorizer(min_df=2, ngram_range=(1, 2))),
    ("smote", SMOTE(random_state=42)),
    ("clf",   LogisticRegression(max_iter=100_000, random_state=42)),
])

params = {
    "clf__C": [0.01, 0.1, 1, 10],
    "clf__solver": ["lbfgs", "saga"],
}

grid_smote = GridSearchCV(
    pipe_smote,
    params, cv=3, scoring="f1_macro", n_jobs=-1, verbose=1
)
time_start = time.time()
grid_smote.fit(X_train, y_train)
time_end = time.time()
print(f"Grid search time: {time_end - time_start} seconds")

best_params = grid_smote.best_params_
print(f"Best parameters: {best_params}")

best_estimator_smote = grid_smote.best_estimator_

In [ ]:
# F1-Macro on validation set

y_val_pred = best_estimator_smote.predict(X_val)
val_f1_smote = f1_score(y_val, y_val_pred, average="macro")   # (verdad, predicción)
print(f"Macro-F1 on validation: {val_f1_smote:.4f}")

print(classification_report(y_val, y_val_pred, digits=3, target_names=le.classes_))

## 1.3 COMPARISON TF-IDF

In [ ]:
# Comparison

f1_tfidf = pd.DataFrame({
    "F1-Macro Val": [val_f1, val_f1_smote],
    "Model Name": ["LR (TF-IDF)", "LR (TF-IDF + SMOTE)"],
}).set_index("Model Name").sort_values("F1-Macro Val", ascending=False)

print("\nComparison of F1-Macro scores on validation set:")
print(f1_tfidf.to_markdown())

plt.figure(figsize=(8, 6))
sns.barplot(data=f1_tfidf, x="Model Name", y="F1-Macro Val", color="salmon")
bars = plt.gca().patches
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.01, round(yval, 4), ha="center", va="bottom", fontsize=10, fontweight="bold")
sns.despine()
plt.title("Comparison of F1-Macro scores on validation set", fontsize=16, fontweight="bold", pad=20)
plt.xlabel("")
plt.show()


- Introducing SMOTE technique did not improve significantly F1-Macro metric, for this reason no SMOTE (synthetic data) technique is selected.

# 2. BETO

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback,
                          DataCollatorWithPadding, set_seed)
from datasets import Dataset
from fake_news_co.paths import MODELS

set_seed(42)
MODEL = "dccuchile/bert-base-spanish-wwm-cased"
MAXLEN = 64          # del EDA: máx subword = 52 -> 64 cubre el 100%

tok = AutoTokenizer.from_pretrained(MODEL)
def to_ds(split):
    d = df[df["split"] == split]
    ds = Dataset.from_dict({"text": d["text"].tolist(), "labels": d["label_id"].tolist()})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=MAXLEN), batched=True)

ds_train, ds_val = to_ds("train"), to_ds("val")


In [ ]:
# pesos inverse-frequency desde el train (Verdadero pesa ~10x)
import torch


counts = np.bincount(df.query("split=='train'")["label_id"], minlength=3)
class_weights = torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=3,
    id2label={i: c for i, c in enumerate(le.classes_)},
    label2id={c: i for i, c in enumerate(le.classes_)})

class WeightedTrainer(Trainer):
    """El Trainer de HF ignora los class weights → se override compute_loss."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            out.logits, labels, weight=class_weights.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {"f1_macro": f1_score(p.label_ids, preds, average="macro"),
            "acc": accuracy_score(p.label_ids, preds)}

args = TrainingArguments(
    output_dir=str(MODELS / "beto"),
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="f1_macro", greater_is_better=True,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    num_train_epochs=6, learning_rate=2e-5, warmup_ratio=0.1, weight_decay=0.01,
    fp16=False, bf16=False,          # ⚠️ MPS no soporta fp16
    report_to="none", logging_steps=25, seed=42)

trainer = WeightedTrainer(
    model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
    processing_class=tok, data_collator=DataCollatorWithPadding(tok),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])

In [ ]:
trainer.train()
print(trainer.evaluate())

In [ ]:
FINAL = MODELS / "beto-final"
trainer.save_model(FINAL)          # pesos + config
tok.save_pretrained(FINAL)         # tokenizer (para poder reusarlo solo)
print("guardado en:", FINAL)

In [ ]:
# FINAL TEST EVALUATION — loads the SAVED model, so it runs without retraining
from sklearn.metrics import (f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

FINAL = MODELS / "beto-final"
tok_final = AutoTokenizer.from_pretrained(FINAL)
beto_final = AutoModelForSequenceClassification.from_pretrained(FINAL)
device = "mps" if torch.backends.mps.is_available() else "cpu"
beto_final.to(device).eval()

@torch.no_grad()
def beto_predict(texts, bs=64):
    preds = []
    for i in range(0, len(texts), bs):
        enc = tok_final(texts[i:i+bs], truncation=True, max_length=MAXLEN,
                        padding=True, return_tensors="pt").to(device)
        preds.append(beto_final(**enc).logits.argmax(-1).cpu().numpy())
    return np.concatenate(preds)

test_df = df[df["split"] == "test"]
y_true = test_df["label_id"].to_numpy()
y_pred = beto_predict(test_df["text"].tolist())

print("Macro-F1 (test):", round(f1_score(y_true, y_pred, average="macro"), 4))
print(classification_report(y_true, y_pred, target_names=le.classes_, digits=3))

# bootstrap CI: Verdadero has only 14 test examples -> report uncertainty honestly
rng = np.random.default_rng(42)
boot = [f1_score(y_true[i], y_pred[i], average="macro")
        for i in (rng.integers(0, len(y_true), len(y_true)) for _ in range(1000))]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Macro-F1 95% CI: [{lo:.3f}, {hi:.3f}]")

ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred),
                       display_labels=le.classes_).plot(cmap="Blues")
plt.show()

In [ ]:
te = df[df["split"] == "test"].copy()
te["true"], te["pred"] = le.inverse_transform(y_true), le.inverse_transform(y_pred)
errs = te[te["true"] != te["pred"]]
print(f"{len(errs)} errores de {len(te)}")
errs[["text", "true", "pred"]].head(15)

# 3. CONCLUSION

| Model | macro-F1 (val) | macro-F1 (test) | Falso F1 (test) | Cuestionable F1 (test) | Verdadero F1 (test) |
|---|---|---|---|---|---|
| LR (TF-IDF, class_weight) | 0.398 | 0.386 | 0.743 | 0.351 | 0.065 |
| **BETO (weighted loss)** | **0.449** | **0.405** | **0.806** | **0.410** | 0.000 |

- **BETO beats the TF-IDF baseline** on macro-F1 (+0.05 val, +0.02 test) and clearly on the two majority classes: `Falso` 0.806 and `Cuestionable` 0.410 on test.
- **`Verdadero` collapses on test (F1 = 0.0)**: all 14 test examples were misclassified (9 → Falso, 5 → Cuestionable), despite the ~10x class weight. On val it only reached 0.143 (2/14). This is the expected consequence of the **structural scarcity** documented in the EDA (93 total examples, declining over time — 33/year in 2020 → 2 in 2026), not a tuning failure.
- Bootstrap **95% CI for BETO test macro-F1: [0.371, 0.440]** — wide, driven by the tiny minority class.
- **Honest read:** with ~10-word claims and 65 training examples of `Verdadero`, none of the tried variants learns that class. In practice the model is a strong `Falso` vs `Cuestionable` discriminator that never predicts `Verdadero` reliably. This limitation is stated in the [Model Card](../docs/MODEL_CARD.md) — presenting this system as a "truth detector" would be a misuse.